# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mhassantahir-afk/ML-Engineering-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane as an ML task is a Ranking or Scoring.
Because the My work will deliver a ranked list of pages 

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Prediction:
Whether a given page is currently declining or not. The model's 
predicted probability is then used to rank all pages, so the pages the model is most confident 
are declining appear first in the review queue.

Proxy Label:
declining_flag (trend_direction == "down", which corresponds to trend_pct <= -20)

This is a proxy label, not a clean observed future outcome. trend_direction is a rule-based 
category computed from trend_pct within the same 90-day window as my features, checking the 
data confirms FlyRank's own cutoff is trend_pct <= -20 for "down" (min -100, max -20 among 
"down" rows). So it is a snapshot bucket rather than a true future-window outcome. Because 
trend_direction is directly derived from trend_pct, neither trend_direction nor trend_pct will 
be used as a feature. Only the label itself.

For now I will be using FlyrankAI's existing threshold (trend_pct <= -20) as is rather than defining a custom cut-off. I however do plan on revisiting this threshold later (e.g testing -30 or -40 or -10) during the signal audit or 
baseline stages, once I can properly compare precision@K and row counts across alternatives. 
A stronger future version of this label would also measure decline in a genuinely separate 
future time window using the warehouse's daily data with real dates, rather than a single 
snapshot from the starter CSV.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My metric is Precision@50. As 50 pages is a realistic amount that can be reviewed/rewritten/refreshed by an editor. 

Since the Precision@50 tree at depth 2 was 0.600 and The hand rule scored 0.680, my model scored 0.720 at depth 3, a good number would mean beating the hand rule means Precision@50 above 0.680.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one page (per client). Each row is a single pseudonymized content item 
(content_id), belonging to one client (client_id), with its own trend_direction and other 
90-day metrics. The dataframe below confirms this as 30,000 rows, no duplicate content_id 
values, one page per row.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd

notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, "..", ".."))
csv_path = os.path.join(project_root, "data", "raw", "content_refresh_anonymized.csv")

df = pd.read_csv(csv_path)

print(f"Total rows: {len(df)}")
print(f"Unique content_id values: {df['content_id'].nunique()}")

# Grain check: if these two numbers match, each row really is one unique page
df[['content_id', 'client_id', 'trend_direction', 'impressions_90d']].head()

Total rows: 30000
Unique content_id values: 30000


,content_id,client_id,trend_direction,impressions_90d
0,content_304f48230142,client_f369cb89fc,down,3803
1,content_a1fb4e703a9e,client_4e07408562,down,15320
2,content_9aa793d4d895,client_7f2253d7e2,down,12581
3,content_331d6c4de07b,client_19581e27de,stable,11751
4,content_d99b7a2d90ca,client_3fdba35f04,down,19140


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule fails here. Impressions alone fail as a signal impressions_90d for declining 
pages (4,919 avg) is nearly identical to pages trending up (4,716 avg), so a threshold on 
impressions cannot separate the two groups. Position also fails on its own because the position of 
declining pages (median ~11.3) is not considerably worse than pages trending up, so a threshold 
on position alone would misclassify many pages too.

It takes a combination of these signals together to reliably tell 
declining pages apart from stable or growing ones. This is exactly the kind of pattern a fixed 
if-statement cannot capture, but a trained model can learn directly from the data. Thus ML is 
required here.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.